# 손글씨 이미지로 배우는 파일 처리와 NumPy

**Google Colab · 학생 탐구형 실습 노트북** (고등학교 1학년 · 파이썬 모듈 활용 단원)

> **핵심 질문:** 컴퓨터는 손글씨 숫자를 어떻게 **데이터**로 표현하고 처리할까?

| 차시 | 주제 | 이 차시가 끝나면 나는… |
|---|---|---|
| 1 | 이미지 파일은 어떤 데이터일까? | 파일을 바이너리로 읽고, 파일 속 바이트와 픽셀이 다르다는 것을 설명할 수 있다 |
| 2 | 픽셀값을 2차원 리스트로 보기 | 픽셀을 `[행][열]`로 읽고 바꿀 수 있다 |
| 3 | 리스트 vs NumPy 배열 ① 구조 | `shape`, `dtype`, 인덱싱·슬라이싱의 차이를 설명할 수 있다 |
| 4 | 리스트 vs NumPy 배열 ② 반전 | 반복문 없이 배열 전체를 계산하는 **벡터화 연산**을 쓸 수 있다 |
| 5 | 조건으로 픽셀 고르기 (필터) | 자르기·이진화·밝기 조절을 NumPy로 할 수 있다 |
| 6 | 컬러(RGB) 이미지 | 3차원 배열을 이해하고 나만의 필터를 만들 수 있다 |

**이 노트북에서 쓰는 모듈**

| 모듈 | 하는 일 |
|---|---|
| `PIL` (Pillow) | 이미지 파일 열기·저장 |
| `numpy` | 숫자 배열 계산 |
| `matplotlib.pyplot` | 그림(이미지) 화면에 보여주기 |
| `time`, `random`, `urllib.request` | 시간 재기, 난수 만들기, 인터넷에서 파일 받기 |

**오류가 나면?** 당황하지 말고 아래 순서로 기록해 보세요.
`예상한 결과 → 실제 결과(오류 메시지) → 원인 추정 → 수정한 코드`

**표시 안내:** ✏️ = 직접 코드를 작성하는 칸 · 🤔 = 생각해 보고 답을 적는 칸 · 💡 = 핵심 정리

## 준비: 실습용 손글씨 이미지 만들기

아래 셀을 **한 번만** 실행하면 `digit.png` 파일이 만들어집니다.
손글씨 숫자 데이터(MNIST)를 인터넷에서 내려받기 때문에 **처음 실행할 때 1분 정도** 걸릴 수 있어요.

> `sklearn`(사이킷런)은 데이터를 내려받는 데만 사용합니다. 이번 수업에서 따로 배우지는 않아요.

In [ ]:
from sklearn.datasets import fetch_openml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 손글씨 숫자 데이터 내려받기 (시간이 조금 걸려요)
mnist = fetch_openml("mnist_784", version=1, as_frame=False)

image = mnist.data[0].reshape(28, 28)   # 첫 번째 손글씨를 28×28 모양으로
label = mnist.target[0]                  # 이 손글씨가 나타내는 숫자

plt.imshow(image, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()
print("숫자:", label)
print("이미지 크기:", image.shape)

# 파일로 저장 → 다음 차시부터 이 파일을 사용합니다
Image.fromarray(image.astype(np.uint8)).save("digit.png")
print("digit.png 저장 완료!")

## 1차시. 이미지 파일은 어떤 데이터일까?

**🤔 예상하기** (실행 전에 먼저 적어 보세요)
1. `read()`로 읽은 결과의 자료형은 무엇일까요?
2. PNG 파일의 첫 번째 바이트가 곧 첫 번째 픽셀의 밝기일까요?

- 내 예상:

In [ ]:
with open("digit.png", "rb") as f:   # 'rb' = read(읽기) + binary(바이너리)
    raw = f.read()

print("자료형:", type(raw))
print("파일 크기:", len(raw), "bytes")
print("처음 20바이트:", raw[:20])
print("정수값으로 보기:", list(raw[:20]))

`bytes`는 0~255 사이의 정수가 줄지어 있는 자료형입니다. 반복문으로 하나씩 꺼내 보면 정수가 나와요.

In [ ]:
data = b"PNG"

for value in data:
    print(chr(value), value, type(value))

**🤔 생각해 보기**
1. 처음 몇 바이트를 글자로 바꾸면 `PNG`가 보입니다. 파일 맨 앞에 왜 이런 정보가 있을까요?
2. 이 이미지는 28 × 28 = **784개**의 픽셀로 되어 있습니다. 그런데 파일 크기는 몇 바이트였나요? 픽셀 수보다 **작다면** 그 이유는 무엇일까요?
   *(힌트: PNG는 데이터를 **압축**해서 저장합니다.)*

- 답:

💡 **정리:** 파일 속 바이트를 그대로 읽는다고 픽셀값이 나오지는 않습니다.
픽셀값을 보려면 PNG 형식을 해석해 주는 **모듈(Pillow)** 이 필요해요. → 2차시

### ✏️ 도전 1 — 예외 처리
없는 파일(`missing.png`)을 읽어도 프로그램이 멈추지 않도록 `try / except FileNotFoundError`를 사용해 보세요.
파일이 없으면 `"파일을 찾을 수 없어요: missing.png"`라고 출력되게 만듭니다.

In [ ]:
# ✏️ missing.png 파일을 바이너리 모드로 읽고, 파일이 없을 때를 예외 처리하기
# try:
#     ...
# except FileNotFoundError:
#     ...

## 2차시. 이미지의 픽셀값 확인하기

`digit.png`는 28×28 크기의 흑백 이미지입니다. 컴퓨터 안에서는 어떤 숫자들로 표현되어 있을까요?

- 흑백 이미지의 픽셀값은 **0(검정) ~ 255(흰색)** 사이의 정수입니다.
- Pillow의 `img.getpixel((x, y))`는 **가로 x번째, 세로 y번째** 픽셀값을 알려 줍니다.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("digit.png").convert("L")   # 'L' = 흑백(밝기 1개) 모드

# 2차원 리스트 만들기: 한 줄(row)씩 만들어서 pixels에 붙인다
pixels = []
for y in range(28):          # 세로 방향 = 행
    row = []
    for x in range(28):      # 가로 방향 = 열
        row.append(img.getpixel((x, y)))
    pixels.append(row)

for row in pixels:
    print(row)

> 참고: 위 코드는 리스트 컴프리헨션으로 한 줄에 쓸 수도 있어요.
> `pixels = [[img.getpixel((x, y)) for x in range(28)] for y in range(28)]`

**🤔 생각해 보기:** 출력된 숫자들 속에서 손글씨 숫자의 모양이 보이나요? 0이 아닌 값들은 어디에 모여 있나요?

### 리스트 값을 바꾸면 이미지도 바뀔까?
14번째 **행**(가로줄)의 모든 값을 255(흰색)로 바꿔 봅시다.

In [ ]:
for x in range(28):
    pixels[14][x] = 255      # pixels[행][열]

plt.imshow(pixels, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()

**🤔 질문:** 이미지 좌표 `(x, y)`와 리스트 인덱스 `pixels[행][열]`은 어떻게 대응할까요?
`img.getpixel((x, y))` 와 같은 값은 `pixels[ ? ][ ? ]` 입니다.

- 답:

### ✏️ 도전 2-1 — 세로줄 그리기
이번에는 **14번째 열**에 흰색 세로줄을 그려 보세요. (원본 이미지를 다시 불러와서 시작합니다.)

In [ ]:
img = Image.open("digit.png").convert("L")
pixels = [[img.getpixel((x, y)) for x in range(28)] for y in range(28)]

# ✏️ 14번째 열의 모든 값을 255로 바꾸기
for y in range(28):
    pixels[____][____] = 255

plt.imshow(pixels, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()

### ✏️ 도전 2-2 — 잉크 픽셀 세기
손글씨가 지나간 픽셀(값이 0보다 큰 픽셀)은 모두 몇 개일까요? 이중 반복문으로 세어 보세요.
*(이 문제는 5차시에서 NumPy로 **한 줄**에 다시 풀어 봅니다.)*

In [ ]:
img = Image.open("digit.png").convert("L")
pixels = [[img.getpixel((x, y)) for x in range(28)] for y in range(28)]

count = 0
# ✏️ 이중 반복문으로 0보다 큰 픽셀 개수 세기


print("잉크 픽셀 수:", count)

## 3차시. NumPy 배열은 리스트와 무엇이 다를까? ① 구조

같은 픽셀을 **2차원 리스트**와 **NumPy 배열(ndarray)** 로 각각 저장하고 비교해 봅시다.
`np.array(img)` 한 줄이면 이미지가 바로 NumPy 배열이 됩니다.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

img = Image.open("digit.png").convert("L")

pixels_list = [[img.getpixel((x, y)) for x in range(28)] for y in range(28)]
pixels_array = np.array(img)

print("=== 2차원 리스트 ===")
print("자료형:", type(pixels_list))
print("행 개수:", len(pixels_list))
print("열 개수:", len(pixels_list[0]))
print("원소 자료형:", type(pixels_list[0][0]))

print("\n=== NumPy 배열 ===")
print("자료형:", type(pixels_array))
print("배열 크기(shape):", pixels_array.shape)
print("차원 수(ndim):", pixels_array.ndim)
print("원소 자료형(dtype):", pixels_array.dtype)

💡 **`uint8`이란?** **u**nsigned(부호 없는) **int**eger(정수) **8**비트.
8비트 = 1바이트이므로 **0~255**까지만 저장할 수 있어요. 픽셀 밝기를 담기에 딱 맞는 크기입니다.

### 원소 꺼내기: 인덱싱과 슬라이싱

In [ ]:
# 같은 픽셀 꺼내기
print("리스트 :", pixels_list[7][10])
print("NumPy  :", pixels_array[7, 10])      # [행, 열] 을 한 번에

# 한 행(가로줄) 꺼내기 — 둘 다 쉽다
print("리스트 7행 :", pixels_list[7])
print("NumPy 7행  :", pixels_array[7])

# 한 열(세로줄) 꺼내기 — 리스트는 반복문이 필요하다
column_list = [row[14] for row in pixels_list]
column_array = pixels_array[:, 14]           # ':' = 모든 행
print("리스트 14열:", column_list)
print("NumPy 14열 :", column_array)

### 메모리는 얼마나 쓸까?
**한 행(28개 픽셀)** 만 비교해 봅시다.

In [ ]:
import sys

row_list = pixels_list[7]
row_array = pixels_array[7]

print("리스트 한 행이 차지하는 크기:", sys.getsizeof(row_list), "bytes")
print("NumPy 한 행의 픽셀 데이터   :", row_array.nbytes, "bytes")
print("NumPy 전체 픽셀 데이터      :", pixels_array.nbytes, "bytes  (28 × 28 × 1바이트)")

💡 리스트는 숫자를 직접 담지 않고, 각 숫자 **객체가 있는 위치(주소)** 를 담습니다. 주소 하나가 8바이트라서 28개면 200바이트가 넘어요. 정수 객체 자체가 차지하는 메모리는 여기에 포함되지도 않았습니다.
NumPy 배열은 `uint8` 숫자를 **한 줄로 붙여서** 1바이트씩 저장하기 때문에 28바이트면 충분합니다.

**🤔 생각해 보기:** 사진 한 장이 1000 × 1000 픽셀이라면, 리스트와 NumPy 중 무엇이 유리할까요?

- 답:

## 4차시. NumPy 배열은 리스트와 무엇이 다를까? ② 검정과 흰색 반전하기

**목표:** 모든 픽셀을 `255 - 기존값`으로 바꿔 검정과 흰색을 뒤집습니다.
먼저 **리스트 + 반복문**으로, 다음은 **NumPy**로 풀고 두 방법을 비교합니다.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

img = Image.open("digit.png").convert("L")

pixels_list = [[img.getpixel((x, y)) for x in range(28)] for y in range(28)]   # 2차원 리스트
pixels_array = np.array(img)                                                  # NumPy 배열

### ✏️ 방법 1 — 리스트와 이중 반복문
`pixels_list`의 값을 이용해 반전된 새 리스트 `inverted_list`를 만드세요.
*힌트: 2차시처럼 `row`를 만들어서 `inverted_list`에 붙입니다.*

In [ ]:
inverted_list = []

# ✏️ 이중 반복문으로 inverted_list 만들기
for y in range(28):
    row = []
    for x in range(28):
        row.append(____________)
    inverted_list.append(row)

plt.imshow(inverted_list, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()

### ✏️ 방법 2 — NumPy
반복문 **없이** 한 줄로 반전 이미지를 만들어 보세요.

In [ ]:
# ✏️ NumPy 배열 전체에 한 번에 연산 적용하기
inverted_np = ____________

plt.imshow(inverted_np, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()

### 실험: 리스트와 NumPy에 같은 계산을 해 보면?
**🤔 실행 전에 예상해 보세요.** `[10, 20, 30] + 10` 과 `[10, 20, 30] * 2` 의 결과는?

In [ ]:
a = [10, 20, 30]

print("리스트 * 2 :", a * 2)

try:
    print("리스트 + 10:", a + 10)
except TypeError as e:
    print("리스트 + 10: 오류!", e)

In [ ]:
import numpy as np

a = np.array([10, 20, 30])

print("NumPy + 10 :", a + 10)     # 각 원소에 10 더하기
print("NumPy * 2  :", a * 2)      # 각 원소에 2 곱하기
print("255 - NumPy:", 255 - a)    # 각 원소를 255에서 빼기

b = np.array([[10, 20, 30],
              [40, 50, 60]])
print("2차원 + 10 :\n", b + 10)   # 2차원 배열도 모든 원소에 적용

리스트의 모든 원소에 10을 더하려면 반복문(또는 리스트 컴프리헨션)이 필요합니다.

In [ ]:
a = [10, 20, 30]
result = []
for value in a:
    result.append(value + 10)
print(result)

# 같은 코드를 한 줄로: [value + 10 for value in a]

💡 **벡터화 연산(Vectorized operation)**
NumPy 배열은 같은 자료형의 원소들을 모아 저장하고, `+`, `-`, `*` 같은 연산을 **모든 원소에 똑같이** 적용합니다.
리스트는 원소를 하나씩 꺼내 계산해야 하지만, NumPy는 **배열 전체에 대한 식 하나**만 쓰면 됩니다.

### 얼마나 빠를까? — `time` 모듈로 재 보기
두 방법으로 반전을 **1000번** 반복하고 걸린 시간을 비교합니다.

In [ ]:
import time

start = time.time()
for _ in range(1000):
    result = [[255 - v for v in row] for row in pixels_list]
list_time = time.time() - start

start = time.time()
for _ in range(1000):
    result = 255 - pixels_array
numpy_time = time.time() - start

print(f"리스트: {list_time:.4f}초")
print(f"NumPy : {numpy_time:.4f}초")
print(f"NumPy가 약 {list_time / numpy_time:.0f}배 빠릅니다")

**🤔 생각해 보기:** 이미지가 28×28이 아니라 스마트폰 사진(약 4000×3000)이라면 차이는 어떻게 될까요?

- 답:

## 5차시. 조건으로 픽셀 고르기 — 간단한 필터 만들기

사진 앱의 '필터'도 결국 **픽셀값을 규칙에 따라 바꾸는 것**입니다.
NumPy로 자르기, 조건 검사, 이진화, 밝기 조절을 해 봅시다.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

digit = np.array(Image.open("digit.png").convert("L"))

# 여러 이미지를 나란히 보여주는 함수
def show_images(images, titles):
    plt.figure(figsize=(3 * len(images), 3))
    for i in range(len(images)):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(images[i], cmap="gray", vmin=0, vmax=255)
        plt.title(titles[i])
        plt.axis("off")
    plt.show()

### ① 자르기 (슬라이싱)
`배열[시작행:끝행, 시작열:끝열]` 로 원하는 부분만 잘라냅니다. (끝 번호는 포함하지 않아요.)

In [ ]:
cropped = digit[4:24, 4:24]
print("자른 크기:", cropped.shape)

show_images([digit, cropped], ["원본 28×28", "자른 이미지 20×20"])

### ② 조건 검사와 개수 세기
배열에 비교 연산을 하면 **원소마다** `True / False`가 나옵니다.

In [ ]:
small = np.array([0, 30, 200, 255, 0])
print(small > 0)            # 원소마다 비교
print((small > 0).sum())    # True는 1, False는 0으로 더해진다

# 2차시 도전 2-2를 NumPy로 한 줄에!
print("잉크 픽셀 수:", (digit > 0).sum())

**🤔 비교:** 2차시에서 반복문으로 센 개수와 같은가요?

### ③ 이진화 — 흑과 백 두 가지로만
`np.where(조건, 참일 때 값, 거짓일 때 값)` 은 원소마다 조건을 검사해서 값을 고릅니다.

In [ ]:
binary = np.where(digit >= 128, 255, 0)

show_images([digit, binary], ["원본", "이진화 (기준 128)"])

### ④ 밝기 조절과 **오버플로우**
모든 픽셀에 100을 더하면 더 밝아질까요? **🤔 먼저 예상해 보세요.**

In [ ]:
a = np.array([100, 150, 200], dtype=np.uint8)
print(a + 100)      # 200 + 100 = 300 ?

`uint8`은 0~255만 담을 수 있어서, 255를 넘으면 **0부터 다시** 셉니다. (300 → 300 - 256 = 44)
이것을 **오버플로우(overflow)** 라고 해요. 자동차 주행거리계가 999999 다음에 000000이 되는 것과 비슷합니다.

**해결 방법:** 더 큰 자료형으로 바꿔서 계산 → `np.clip`으로 0~255 범위에 맞추기 → 다시 `uint8`로

In [ ]:
wrong = digit + 100                                            # 오버플로우 발생
temp = digit.astype(np.int16)                                  # 더 큰 정수형으로 변환
right = np.clip(temp + 100, 0, 255).astype(np.uint8)           # 0~255로 자르기

show_images([digit, wrong, right], ["원본", "그냥 +100 (오류)", "clip 사용"])

### ✏️ 도전 5 — 기준값 비교하기
이진화 기준값을 **50, 128, 200** 으로 바꿔 가며 세 이미지를 나란히 출력하세요.
기준값이 커지면 숫자 모양이 어떻게 달라지나요?

In [ ]:
# ✏️ 기준값 50, 128, 200으로 이진화한 이미지 만들기
binary_50 = ____________
binary_128 = ____________
binary_200 = ____________

show_images([binary_50, binary_128, binary_200], ["기준 50", "기준 128", "기준 200"])

- 관찰한 점:

## 6차시. 컬러 이미지는 무엇이 다를까?

컬러 이미지는 픽셀 하나가 **빨강(R) · 초록(G) · 파랑(B)** 세 개의 값을 가집니다.
그래서 배열 모양이 `(세로, 가로, 3)` 인 **3차원 배열**이 됩니다.

### 무작위 색으로 작은 컬러 이미지 만들기
리스트로 만들면 반복문이 **세 겹**이 됩니다.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

pixels_list = []
for y in range(28):              # 행 (세로)
    row = []
    for x in range(28):          # 열 (가로)
        rgb = []
        for c in range(3):       # 채널 R, G, B
            rgb.append(random.randint(0, 255))
        row.append(rgb)
    pixels_list.append(row)

print("첫 번째 픽셀의 [R, G, B]:", pixels_list[0][0])

pixels_array = np.array(pixels_list, dtype=np.uint8)
print("shape:", pixels_array.shape)
print("dtype:", pixels_array.dtype)

plt.imshow(pixels_array)
plt.axis("off")
plt.show()

**🤔 비교:** NumPy로는 같은 이미지를 한 줄로 만들 수 있어요. 결과의 `shape`가 같은지 확인해 보세요.

In [ ]:
random_image = np.random.randint(0, 256, size=(28, 28, 3), dtype=np.uint8)
print("shape:", random_image.shape)

### 실제 컬러 사진 불러오기

In [ ]:
import urllib.request
from PIL import Image

url = "https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/coffee.png"
urllib.request.urlretrieve(url, "coffee.png")

img = Image.open("coffee.png").convert("RGB")
photo = np.array(img)

print("이미지 크기 (가로, 세로):", img.size)
print("배열 형태 (세로, 가로, 채널):", photo.shape)
print("자료형:", photo.dtype)
print("왼쪽 위 첫 픽셀 [R, G, B]:", photo[0, 0])

# 이미지 출력 함수 (흑백이면 gray로)
def show_image(image):
    plt.imshow(image, cmap="gray" if image.ndim == 2 else None)
    plt.axis("off")
    plt.show()

show_image(photo)

**🤔 질문:** `img.size`는 `(가로, 세로)` 순서인데 `photo.shape`는 `(세로, 가로, 3)` 순서입니다. 2차시의 `(x, y)` ↔ `[행][열]` 관계와 어떻게 연결되나요?

- 답:

### 채널 하나만 바꿔 보기
`photo[:, :, 0]` = 모든 행, 모든 열의 **0번 채널(R)**

In [ ]:
no_red = photo.copy()      # 원본을 지키기 위해 복사본 사용
no_red[:, :, 0] = 0        # R 채널을 모두 0으로

show_image(no_red)

### 4·5차시에서 배운 연산을 컬러 사진에 적용하기

In [ ]:
# 반전 (4차시)
show_image(255 - photo)

# 밝게 (5차시 — 오버플로우 주의!)
bright = np.clip(photo.astype(np.int16) + 50, 0, 255).astype(np.uint8)
show_image(bright)

### 흑백으로 바꾸기
픽셀마다 R, G, B 세 값의 **평균**을 내면 밝기 하나가 됩니다.
`axis=2`는 **세 번째 축(채널 방향)** 으로 평균을 내라는 뜻이에요. → `(세로, 가로, 3)` 이 `(세로, 가로)` 가 됩니다.

In [ ]:
gray = np.mean(photo, axis=2).astype(np.uint8)
print("흑백 shape:", gray.shape)
show_image(gray)

binary = np.where(gray >= 128, 255, 0).astype(np.uint8)   # 5차시 이진화
show_image(binary)

### ✏️ 도전 6 — 나만의 필터 만들기
지금까지 배운 기술 중 **두 가지 이상을 조합**해서 나만의 필터를 만들어 보세요.

| 쓸 수 있는 기술 | 코드 예 |
|---|---|
| 자르기 | `photo[100:300, 150:450]` |
| 반전 | `255 - photo` |
| 밝기 조절 | `np.clip(photo.astype(np.int16) + 값, 0, 255).astype(np.uint8)` |
| 채널 바꾸기 | `photo[:, :, 1] = 0` (복사본에서!) |
| 채널 순서 뒤집기 | `photo[:, :, ::-1]` (RGB → BGR) |
| 흑백·이진화 | `np.mean(..., axis=2)`, `np.where(...)` |

In [ ]:
my_filter = photo.copy()

# ✏️ 나만의 필터 코드 작성


show_image(my_filter)

- 내 필터 이름:
- 사용한 기술과 순서:

## 정리 및 성찰

| 개념 | 한 줄 정리 |
|---|---|
| 바이너리 파일 | 파일은 0~255 바이트의 나열이고, PNG는 압축되어 있어 바이트 ≠ 픽셀 |
| 픽셀 | 흑백은 밝기 1개(0~255), 컬러는 R·G·B 3개 |
| 2차원 리스트 | `pixels[행][열]`, 계산하려면 반복문 필요 |
| NumPy 배열 | `shape`, `dtype`, `배열[행, 열]`, 슬라이싱 `[:, 14]` |
| 벡터화 연산 | `255 - 배열`처럼 식 하나로 모든 원소 계산, 빠르고 메모리 효율적 |
| uint8 오버플로우 | 255를 넘으면 0부터 다시 → `astype` + `np.clip`으로 해결 |

**🤔 성찰 질문**
1. 이번 실습에서 가장 새롭게 알게 된 점은?
2. 리스트 대신 NumPy를 써야 하는 상황을 한 가지 예로 들어 보세요.
3. 기록한 오류 중 하나를 골라 `예상 → 실제 → 원인 → 수정` 순서로 정리해 보세요.

- 답: